# Gaia DR3 open-cluster members → CSV

Downloads Gaia DR3 data for a chosen open cluster, selects member candidates by proper motion and parallax, and writes a CSV file ready for a colour–magnitude diagram (CMD) in Excel, Python, etc.

Steps:

1. Gaia DR3 ADQL cone search around the cluster centre (G < 18.5), pre-filtered by proper motion and parallax (5σ around literature values).
2. Membership selection: |μα* − μα*,0| < 3σ_PM, |μδ − μδ,0| < 3σ_PM, |ϖ − ϖ₀| < 3σ_ϖ, RUWE < 1.4.
3. Intrinsic colour (BP−RP)₀ and absolute magnitude M_G, corrected for distance modulus and extinction.
4. CSV output with columns:
   - `Source_ID` – Gaia DR3 source identifier
   - `RA_deg`, `Dec_deg` – position (degrees)
   - `Parallax_mas` – parallax (mas)
   - `PMra_mas_yr`, `PMdec_mas_yr` – proper motion (mas/yr)
   - `BP_RP_0` – intrinsic colour (mag)
   - `M_G` – absolute G magnitude (mag)

**Usage:** choose a cluster in the settings cell and run all cells (Runtime → Run all). The CSV is downloaded automatically at the end.

Data: [Gaia DR3](https://www.cosmos.esa.int/gaia) (ESA/Gaia/DPAC), queried through the public Gaia Archive with `astroquery`. Cluster parameters: Gaia Collaboration, Babusiaux et al. (2018, A&A 616, A10) — centre, proper motion and parallax from Tables A.3/A.4, distance modulus and E(B−V) from Table 2 (A_V = 3.1 E(B−V)); literature ages from the reference in each cluster's `age_ref` field. Extinction coefficients: Wang & Chen (2019, ApJ 877, 116, Table 3). RUWE < 1.4 threshold: Lindegren (2018, Gaia technical note GAIA-C3-TN-LU-LL-124-01).

In [ ]:
!pip install -q astroquery

In [ ]:
CLUSTER = "M67"  #@param ["Pleiades", "Hyades", "M67", "NGC2516", "NGC752"]
OUTPUT = ""      #@param {type:"string"}
# Leave OUTPUT empty to use <CLUSTER>_members.csv

In [ ]:
import csv
import warnings

import numpy as np
from astroquery.gaia import Gaia

warnings.filterwarnings("ignore")
Gaia.MAIN_GAIA_TABLE = "gaiadr3.gaia_source"

# ──────────────────────────────────────────────────────────────────────────
# Catalogue of well-known open clusters
# ra/dec: cluster centre [deg]; radius_deg: search radius [deg]
# pmra/pmdec: mean proper motion [mas/yr]; pm_sigma: selection width [mas/yr]
# parallax: mean parallax [mas]; parallax_sigma: selection width [mas]
# distance_pc: distance [pc]; Av: V-band extinction [mag]; age_lit_myr: age [Myr]
#
# Sources ("GC18" = Gaia Collaboration, Babusiaux et al. 2018, A&A 616, A10;
# VizieR J/A+A/616/A10):
#   ra, dec, pmra, pmdec, parallax - GC18, Table A.3 (Pleiades, Hyades) and
#       Table A.4 (M67, NGC 2516, NGC 752); mean values of cluster members.
#   distance_pc - from the distance modulus DM in GC18, Table 2:
#       d = 10^(DM/5 + 1) pc.
#   Av - A_V = 3.1 * E(B-V), with E(B-V) from GC18, Table 2 (R_V = 3.1 as in
#       GC18, Sect. 2.2).
#   age_lit_myr - the source is given in the age_ref field of each cluster.
#   radius_deg, pm_sigma, parallax_sigma - NOT literature values but choices
#       made for this exercise: search radius and selection-window width
#       (a member satisfies |x - x0| < 3 sigma).
# ──────────────────────────────────────────────────────────────────────────
CLUSTERS = {
    "Pleiades": {
        "name": "Pleiades (M45)",
        "ra": 56.44, "dec": 23.86, "radius_deg": 2.5,
        "pmra": 19.997, "pmdec": -45.548, "pm_sigma": 2.0,
        "parallax": 7.364, "parallax_sigma": 0.5,
        "distance_pc": 136.0,   # DM = 5.667
        "Av": 0.140,            # E(B-V) = 0.045
        "age_lit_myr": 112,     # 112 ± 5 Myr
        "age_ref": "Dahm 2015, ApJ 813, 108",
    },
    "Hyades": {
        "name": "Hyades",
        "ra": 67.98, "dec": 17.01, "radius_deg": 6.0,
        "pmra": 101.005, "pmdec": -28.490, "pm_sigma": 4.0,
        "parallax": 21.052, "parallax_sigma": 1.5,
        "distance_pc": 47.6,    # DM = 3.389
        "Av": 0.003,            # E(B-V) = 0.001
        "age_lit_myr": 625,     # 625 ± 50 Myr
        "age_ref": "Perryman et al. 1998, A&A 331, 81",
    },
    "M67": {
        "name": "M67 (NGC 2682)",
        "ra": 132.8476, "dec": 11.8369, "radius_deg": 0.5,
        "pmra": -10.9737, "pmdec": -2.9396, "pm_sigma": 0.5,
        "parallax": 1.1325, "parallax_sigma": 0.15,
        "distance_pc": 881.5,   # DM = 9.726
        "Av": 0.115,            # E(B-V) = 0.037
        "age_lit_myr": 3950,    # 3.95 (+0.16 / -0.15) Gyr
        "age_ref": "Reyes et al. 2024, MNRAS 532, 2860",
    },
    "NGC2516": {
        "name": "NGC 2516",
        "ra": 119.5469, "dec": -60.7749, "radius_deg": 0.8,
        "pmra": -4.6579, "pmdec": 11.1517, "pm_sigma": 0.8,
        "parallax": 2.4118, "parallax_sigma": 0.30,
        "distance_pc": 415.1,   # DM = 8.091
        "Av": 0.220,            # E(B-V) = 0.071
        "age_lit_myr": 150,     # ≈ 150 Myr
        "age_ref": "Bouma et al. 2021, AJ 162, 197",
    },
    "NGC752": {
        "name": "NGC 752",
        "ra": 29.2054, "dec": 37.7454, "radius_deg": 0.7,
        "pmra": 9.8092, "pmdec": -11.7637, "pm_sigma": 0.5,
        "parallax": 2.2304, "parallax_sigma": 0.20,
        "distance_pc": 449.6,   # DM = 8.264
        "Av": 0.124,            # E(B-V) = 0.040
        "age_lit_myr": 1340,    # 1.34 ± 0.06 Gyr
        "age_ref": "Agüeros et al. 2018, ApJ 862, 33",
    },
}

# Extinction coefficients A_band / A_V for the Gaia passbands
# (Wang & Chen 2019, ApJ 877, 116, Table 3)
A_G_PER_AV = 0.789
A_BP_PER_AV = 1.002
A_RP_PER_AV = 0.589

# Magnitude limit of the query - a choice for this exercise, not a literature value
G_MAG_LIMIT = 18.5


def fetch_gaia(cluster):
    """ADQL cone search around the cluster centre, with a coarse proper-motion
    / parallax pre-filter (5 sigma around literature values) so that the server
    returns fewer rows. The finer 3-sigma cut is applied in select_members()."""
    ra, dec, r = cluster["ra"], cluster["dec"], cluster["radius_deg"]
    pmra_lo, pmra_hi = cluster["pmra"] - 5.0 * cluster["pm_sigma"], cluster["pmra"] + 5.0 * cluster["pm_sigma"]
    pmdec_lo, pmdec_hi = cluster["pmdec"] - 5.0 * cluster["pm_sigma"], cluster["pmdec"] + 5.0 * cluster["pm_sigma"]
    plx_lo, plx_hi = cluster["parallax"] - 5.0 * cluster["parallax_sigma"], cluster["parallax"] + 5.0 * cluster["parallax_sigma"]
    print(f"  Gaia DR3 query (RA={ra} deg, Dec={dec} deg, r={r} deg, G<{G_MAG_LIMIT}, "
          f"PM/parallax pre-filter 5 sigma) ...", flush=True)
    adql = f"""
    SELECT source_id, ra, dec,
           parallax, parallax_error,
           pmra, pmra_error, pmdec, pmdec_error,
           phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag, ruwe
    FROM gaiadr3.gaia_source
    WHERE CONTAINS(POINT('ICRS', ra, dec),
                   CIRCLE('ICRS', {ra}, {dec}, {r})) = 1
      AND phot_g_mean_mag IS NOT NULL
      AND phot_bp_mean_mag IS NOT NULL
      AND phot_rp_mean_mag IS NOT NULL
      AND parallax IS NOT NULL
      AND pmra IS NOT NULL AND pmdec IS NOT NULL
      AND phot_g_mean_mag < {G_MAG_LIMIT}
      AND parallax BETWEEN {plx_lo} AND {plx_hi}
      AND pmra BETWEEN {pmra_lo} AND {pmra_hi}
      AND pmdec BETWEEN {pmdec_lo} AND {pmdec_hi}
    """
    job = Gaia.launch_job_async(adql)
    t = job.get_results()
    print(f"  Retrieved {len(t):,} stars in the field.")
    return t


def select_members(t, cl):
    """Select cluster members by proper motion, parallax and astrometric quality.
    RUWE < 1.4 is the threshold for a reliable astrometric solution (Lindegren
    2018, Gaia technical note GAIA-C3-TN-LU-LL-124-01)."""
    mask = (
        (np.abs(t["pmra"]  - cl["pmra"])  < 3.0 * cl["pm_sigma"]) &
        (np.abs(t["pmdec"] - cl["pmdec"]) < 3.0 * cl["pm_sigma"]) &
        (np.abs(t["parallax"] - cl["parallax"]) < 3.0 * cl["parallax_sigma"]) &
        (t["parallax"] > 0) & (t["ruwe"] < 1.4)
    )
    m = t[mask]
    print(f"  Selected {len(m):,} member candidates.")
    return m


def compute_cmd(t, cl):
    """Intrinsic colour (BP-RP)_0 and absolute magnitude M_G."""
    d_pc = cl["distance_pc"]
    Av = cl["Av"]
    mu = 5.0 * np.log10(d_pc / 10.0)   # distance modulus
    A_G  = A_G_PER_AV  * Av
    A_BP = A_BP_PER_AV * Av
    A_RP = A_RP_PER_AV * Av

    G  = np.array(t["phot_g_mean_mag"],  dtype=float)
    BP = np.array(t["phot_bp_mean_mag"], dtype=float)
    RP = np.array(t["phot_rp_mean_mag"], dtype=float)

    M_G   = G - mu - A_G
    color = (BP - A_BP) - (RP - A_RP)
    return color, M_G


def write_csv(path, table, color, M_G, cl):
    with open(path, "w", encoding="utf-8", newline="") as f:
        f.write(f"# Cluster: {cl['name']}\n")
        f.write(f"# Centre (RA, Dec): ({cl['ra']}, {cl['dec']}) deg\n")
        f.write(f"# Search radius: {cl['radius_deg']} deg\n")
        f.write(f"# Distance: {cl['distance_pc']} pc\n")
        f.write(f"# A_V (extinction): {cl['Av']} mag\n")
        f.write(f"# Literature age: {cl['age_lit_myr']} Myr "
                f"(log(age)={np.log10(cl['age_lit_myr']*1e6):.2f}; "
                f"{cl['age_ref']})\n")
        f.write("# Cluster parameters: Gaia Collaboration, Babusiaux et al. "
                "2018, A&A 616, A10\n")
        f.write(f"# Number of members: {len(table)}\n")
        f.write("#\n")
        w = csv.writer(f)
        w.writerow(["Source_ID", "RA_deg", "Dec_deg",
                    "Parallax_mas", "PMra_mas_yr", "PMdec_mas_yr",
                    "BP_RP_0", "M_G"])
        for i in range(len(table)):
            w.writerow([
                int(table["source_id"][i]),
                f"{float(table['ra'][i]):.5f}",
                f"{float(table['dec'][i]):.5f}",
                f"{float(table['parallax'][i]):.4f}",
                f"{float(table['pmra'][i]):.3f}",
                f"{float(table['pmdec'][i]):.3f}",
                f"{float(color[i]):.4f}",
                f"{float(M_G[i]):.4f}",
            ])


def main(cluster, output=None):
    if cluster not in CLUSTERS:
        print(f"Unknown cluster: {cluster}")
        print("Supported clusters:")
        for k, v in CLUSTERS.items():
            print(f"  {k:10s}  {v['name']:18s}  age {v['age_lit_myr']:>5} Myr,"
                  f"  d = {v['distance_pc']:>4.0f} pc")
        return

    cl = CLUSTERS[cluster]
    output = output if output else f"{cluster}_members.csv"

    print(f"Cluster: {cl['name']}")
    t = fetch_gaia(cl)
    m = select_members(t, cl)
    color, M_G = compute_cmd(m, cl)
    write_csv(output, m, color, M_G, cl)
    print(f"Saved: {output}  ({len(m)} members)")

    try:
        from google.colab import files
        files.download(output)
    except ImportError:
        pass  # not running in Colab; the file stays in the working directory

In [ ]:
main(CLUSTER, OUTPUT or None)